In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.interpolate import interpn

import numbers

from thermal_loads_261001 import *

In [ ]:
# environment settings: 
pd.set_option('display.max_column',None)
pd.set_option('display.max_rows',50)
pd.set_option('display.max_seq_items',None)
pd.set_option('display.max_colwidth', 500)
pd.set_option('expand_frame_repr', True)

np._core.arrayprint.set_printoptions(linewidth= 180)
np.set_printoptions(threshold=np.inf)

In [ ]:
red       = (0.984313725490196, 0.5019607843137255, 0.4470588235294118) 
orange    = (0.9921568627450981, 0.7058823529411765, 0.3843137254901961)  
yellow    = (1.0, 0.9294117647058824, 0.43529411764705883)
blue      = (0.09019607843137255, 0.7450980392156863, 0.8117647058823529)
lightblue = (0.6196078431372549, 0.8549019607843137, 0.8980392156862745)  
grey      = (0.83, 0.83, 0.83)

# Data

In [ ]:
project = '291001_TVVL_restaurant'

#### Construction quality level (thermal bridges)
'A' : New building with certified minimization of thermal bridges  
'B' : New building compliant with general codes of practice  
'C' : Building with internal insulation interrupted by solid floor slabs  
'D' : Other buildings  

In [ ]:
building_quality = 'B'

#### Heating system (stratification)   
'HAC' : Hot-air heating without additional destratification   
'HAD' : Hot-air heating with additional destratification   
'CRP' : Ceiling-mounted radiant panels   
'RTH' : Radiant tube heating   
'IRH' : Infrared radiant heater   
'SUR' : Integrated surface heat emission   
'RAD' : Radiators   
'NOT' : No heating system specified

In [ ]:
heating_syst   = 'SUR'

#### Postal code

In [ ]:
CP = 1000

#### Internal temperature

In [ ]:
t_in  = 20  # Base internal temperature 

#### Infiltration rate at 50 Pa

In [ ]:
n_50  = 1.0

#### Windows and doors U-values

In [ ]:
U_wd  = 1.50  #[W/m2/K]
U_dr  = 1.50   #[W/m2/K]

In [ ]:
col_names = ['building_quality', 'heating_syst', 'CP', 't_in', 'n_50', 'U_wd', 'U_dr']
gen_data  = [ building_quality, heating_syst, CP, t_in, n_50, U_wd, U_dr]

df_gen_data = pd.DataFrame({'Names': col_names, 'Values': gen_data})
df_gen_data = df_gen_data.set_index('Names')

#### Additional Power for Reheating

In [ ]:
# Choose between EN12831 Annex F Tables 'F1' or 'F3'
rh_table = 'F1'

# 'L' : Lightweight structure, 'H' : Heavy structure
th_mass = 'H'

# Infiltration rate during setback time (min 0.1 vol/h, max 0.5 vol/h)
n_inf_setback = 0.5

# Heating set back duration (max 168 h for table 'F1')
time_setback_h = 14

# Reheat time in hours (min 0.5 h)
time_reheat_h = 6

# Minimum reheat factor (W/m² of floor heated area)
f_rh_min = 10 # W/m²

In [ ]:
col_names = ['rh_table', 'thermal_mass', 'infiltration_rate_setback', \
             'time_setback_h', 'time_reheat_h', 'f_rh_min']
rh_data = [rh_table, th_mass, n_inf_setback, time_setback_h, time_reheat_h, f_rh_min]

df_rh_data = pd.DataFrame({'Names': col_names, 'Values': rh_data})
df_rh_data = df_rh_data.set_index('Names')

#### Ventilation

In [ ]:
# L: Lenght, B : Breadth, H : Height [m]

L_int = 3.6 * 4
B_int = 11.40
H_int = 3.01

A_floor = L_int * B_int

# Occupants requirements
q_occ_m3h  = 45 # Fresh air flow per occupant(m³/h.occ)

# Floor area per occupant
A_fl_occ = 10 # Office value: A supplementary AHU will be added

# Number of occupants
n_occ      = A_floor / A_fl_occ  # Number of occupants

qv_su_m3h   = n_occ * q_occ_m3h # ventilation fan supply air flow  (m³/h)
qv_ex_m3h   = n_occ * q_occ_m3h # ventilation fan extracted air flow  (m³/h)
epsilon_hx  = 0.80 # efficiency of ventilation system heat recovery exchanger

# Entity, floor heated area, ceiling height, fresh air flow (m³/h), efficiency of heat recovery
vent = \
[('restaurant', A_floor,  H_int,  qv_su_m3h, qv_ex_m3h, epsilon_hx)]

dfv = pd.DataFrame(vent, columns=('zone', 'area_fl', 'h_ceiling', 'q_su_m3h', 'q_ex_m3h', 'epsilon_rec'))
dfv = dfv.set_index('zone')

#### Materials

In [ ]:
materials_list = []

materials_list.append({"material":'concrete_bloc'      ,"lambda":1.36,"rho":2000,"c":1000})
materials_list.append({"material":'insulation'         ,"lambda":0.026,"rho":80, "c":840})
materials_list.append({"material":'reinforced_concrete',"lambda":2.30,"rho":2400,"c":1000})
materials_list.append({"material":'concrete'           ,"lambda":2.10,"rho":2400,"c":1000})
materials_list.append({"material":'brick'              ,"lambda":1.59,"rho":2200,"c":1000})
materials_list.append({"material":'screed'             ,"lambda":1.40,"rho":2000,"c":1000})
materials_list.append({"material":'foamglass'          ,"lambda":0.055,"rho":150,"c":1000})

materials = pd.DataFrame(materials_list)
# materials

#### Wall types layers

In [ ]:
# Wall layers from indoor to outdoor

wt_layers = \
[('wall_ext', 0, 0.190, 'concrete_bloc'),
 ('wall_ext', 1, 0.100, 'insulation'),
 
 ('wall_sep', 0, 0.190, 'concrete_bloc'),
 
 ('roof'    , 0, 0.300, 'reinforced_concrete'),
 ('roof'    , 1, 0.200, 'insulation'),

 ('floor_sep_u', 0, 0.070, 'screed'),
 ('floor_sep_u', 1, 0.300, 'concrete'),
]

wt_layers = pd.DataFrame(wt_layers, columns=('wall_type', 'layer', 'thickness', 'material'))
wt_layers = wt_layers.sort_values(['wall_type', 'layer'])
wt_layers = wt_layers.set_index('wall_type')

# List of wall types
wt_layers.index.unique().tolist()

#### Wall types environments

In [ ]:
# list of wall types in contact with the ground
GRwalls = []

# list of wall types with thermal bridges
TBwalls = ['roof', 'wall_ext']

# list of wall types adjacent to another building entity 
# (not considered for heat production sizing)
BEwalls = ['floor_sep_u']

wt_env = pd.DataFrame({'wall_type': wt_layers.index.unique().tolist()})
wt_env['ground_contact']  = wt_env['wall_type'].isin(GRwalls)
wt_env['thermal_bridges'] = wt_env['wall_type'].isin(TBwalls)
wt_env['adjacent_entity'] = wt_env['wall_type'].isin(BEwalls)

wt_env = wt_env.set_index('wall_type')

#### Wall dimensions and environmental temperatures

In [ ]:
# List of wall types
wt_layers.index.unique().tolist()

In [ ]:
# Wall thicknesses
dftk = (
    wt_layers.groupby(level='wall_type')['thickness']
             .sum()
             .round(2)
             .reset_index()
        )
thk = dict(zip(dftk['wall_type'], dftk['thickness']))
thk

In [ ]:
twle    = thk['wall_ext']
t_floor = thk['floor_sep_u']
t_roof  = thk['roof']

L_rm = L_int + 0.40
B_rm = B_int + 0.35 + 0.40
H_rm = H_int + t_roof + t_floor/2

# Zone, azimuth, slope, wall_type, horiz dim1, horiz dim2, vert dim, h_in, h_out, temp environnement
walls = \
[('restaurant',     0, 180, 'roof',        L_rm,  B_rm,     0,  8, 23, 't_ext'),
 ('restaurant',   180,  90, 'wall_ext',    B_int, 0,    H_int,  8,  8, 't_ext'),
 ('restaurant',   -90,  90, 'wall_ext',    L_rm,  0,     H_rm,  8,  8,  15    ),
 ('restaurant',     0,  90, 'wall_ext',    B_rm,  0,     H_rm,  8,  8, 't_ext'),
 ('restaurant',    90,  90, 'wall_ext',    L_rm,  0,     H_rm,  8,  8, 't_ext'),
 ('restaurant',     0,   0, 'floor_sep_u', L_rm, B_rm,      0,  8,  8, 't_int')]

wl = pd.DataFrame(walls, columns=('zone', 'azimuth', 'slope', 'wall_type', \
                                  'dimh1', 'dimh2', 'dimv', 'h_in', 'h_out', 't_env_name')).set_index('zone')
wl

#### Windows

In [ ]:
# Zone, azimuth, slope, wall_type, number of identical windows, width, height, sill height

H_wd_1 = 3.01
sill_wd_1 = t_floor/2

H_wd_2 = 0.8
sill_wd_2 = 0.8

windows = \
[('restaurant',     0, 90, 'wall_ext', 1, 11.80, H_wd_1, sill_wd_1),
 ('restaurant',    90, 90, 'wall_ext', 1, L_rm,  H_wd_1, sill_wd_1),
 ('restaurant',   -90, 90, 'wall_ext', 4, 1.80,  H_wd_2, sill_wd_2)]

wd = pd.DataFrame(windows, columns=('zone', 'azimuth', 'slope', 'wall_type', \
                                    'number', 'width', 'height', 'sill')).set_index('zone')
# wd

#### Doors

In [ ]:
# Zone, azimuth, slope, wall_type, number of identical external doors, width, height

doors = \
[]

dr = pd.DataFrame(doors, columns=('zone', 'azimuth', 'slope', 'wall_type', \
                                  'number', 'width', 'height')).set_index('zone')
# dr

# Compute Heat Losses

In [ ]:
# dfwt  : Walls thicknesses [m] and conductive resistance [m²K/W]
# dfAU  : Walls U_values [W/m²K] - Temperature Factors - Areas [m²] - Heat Tranfer Coefficients [W/K]
# dfgrAU : Walls Areas [m²] and Heat Tranfer Coefficients [W/K] by zone and wall type
# dfAHT : Wall Areas [m²] and Heat Tranfer Coefficients [W/K] by zone
# dfH   : Total Heat Tranfer Coefficients by zone [W/K]
# dfPHI : Emission and production Heat losses and Loads by zone [W]
# dffv  : Emission Heat losses and loads by zone and by facade [W]
# dffPHI : Emission Heat losses and loads by facade [W]

dfwt, dfAU, dfgrAU, dfAHT, dfH, dfPHI, dffv, dffPHI = thermal_loads(gen_data, rh_data, dfv, \
                  materials, wt_layers, wt_env, wl, wd, dr)

In [ ]:
# Display style for the dataframes
styles = [dict(selector="caption",
                       props=[("text-align", "middle"),
                              ("font-size", "110%"),
                              ("font-weight", "bold"),
                              ("color", 'black'),
                              ("padding-bottom", "10px"),
                              ("white-space", "nowrap")
                             ])]

In [ ]:
numeric_cols = dfAU.select_dtypes(include='number').columns

dfAU.style \
    .format("{:.2f}", subset=numeric_cols) \
    .set_caption("Walls U_values [W/m²K] - Temperature Factors - Areas [m²] - Heat Tranfer Coefficients [W/K]") \
    .set_table_styles(styles)

In [ ]:
numeric_cols = dfgrAU.select_dtypes(include='number').columns

dfgrAU.style \
    .format("{:.2f}", subset=numeric_cols) \
    .set_caption("Walls Areas [m²] and Heat Tranfer Coefficients [W/K] by zone and wall type") \
    .set_table_styles(styles)

In [ ]:
bold_cols = ['area', 'H_T']
bold_cols = [col for col in dfAHT.columns if col.startswith(tuple([x + ' ' for x in bold_cols]))]

dfAHT.style \
    .format("{:.2f}") \
    .set_properties(subset=bold_cols, **{"font-weight": "bold"}) \
    .set_caption("Wall Areas [m²] and Heat Tranfer Coefficients [W/K] by zone") \
    .set_table_styles(styles)

In [ ]:
bold_cols = ['H_T', 'H_V', 'H_TOT']
bold_cols = [col for col in dfH.columns if col.startswith(tuple([x + ' ' for x in bold_cols]))]

dfH.style \
    .format("{:.1f}") \
    .set_properties(subset=bold_cols, **{"font-weight": "bold"}) \
    .set_caption("Total Heat Tranfer Coefficients by zone [W/K]") \
    .set_table_styles(styles)

In [ ]:
bold_cols = ['PHI_T', 'PHI_V', 'PHI_TOT', 'PHI_HL']
bold_cols = [col for col in dfPHI.columns if col.startswith(tuple([x + ' ' for x in bold_cols]))]

dfPHI.style \
    .format("{:.0f}") \
    .set_properties(subset=bold_cols, **{"font-weight": "bold"}) \
    .set_caption("Heat losses and Loads by zone [W]") \
    .set_table_styles(styles)

In [ ]:
bold_cols = ['PHI_TOT', 'PHI_HL']
bold_cols = [col for col in dfPHI.columns if col.startswith(tuple([x + ' ' for x in bold_cols]))]

dfPHI[['PHI_T [W]', 'PHI_V [W]', 'PHI_TOT [W]', 'PHI_RH [W]', 'PHI_HL [W]']].style \
    .format("{:.0f}") \
    .set_properties(subset=bold_cols, **{"font-weight": "bold"}) \
    .set_caption("Heat losses and Loads by zone [W]") \
    .set_table_styles(styles)

In [ ]:
dffv.style \
    .format("{:.0f}") \
    .set_caption("Emission Heat losses and loads by zone and by facade [W]") \
    .set_table_styles(styles)

In [ ]:
dffPHI.style \
    .format("{:.0f}") \
    .set_caption("Emission Heat losses and loads by facade [W]") \
    .set_table_styles(styles)

# Graph

In [ ]:
# Display or not heat production results
display_production = False

In [ ]:
lst_cols = ['PHI_T_wl', 'PHI_T_wd', 'PHI_T_dr', 'PHI_V_su', 'PHI_V_ie', 'PHI_RH']
cols = [col for col in dfPHI.columns if col.startswith(tuple([x + ' ' for x in lst_cols]))]

if display_production :
    df_plot = dfPHI[cols].copy()
else:
    df_plot = dfPHI[cols].iloc[:-1].copy()
    
df_plot.index.name = '' 
df_plot.loc[::-1].plot.barh(figsize=(8,2), 
            stacked=True, color = (red, orange, yellow, blue, lightblue, grey))

plt.xlabel("Heating Power [W]")
plt.title('Breakdown of the zone Heat Loads [W]')
plt.legend(ncol = 1, loc="upper left",
          bbox_to_anchor=(1, 0, 0.5, 1))
plt.grid(linestyle = 'dotted');

In [ ]:
lst_cols = ['PHI_T_wl', 'PHI_T_wd', 'PHI_T_dr', 'PHI_V_su', 'PHI_V_ie', 'PHI_RH']
cols = [col for col in dfPHI.columns if col.startswith(tuple([x + ' ' for x in lst_cols]))]

if display_production :
    df_plot = dfPHI[cols].copy() / A_floor
else:
    df_plot = dfPHI[cols].iloc[:-1].copy() / A_floor
df_plot.index.name = ''

df_plot.loc[::-1].plot.barh(figsize=(8,2), \
            stacked=True, color = (red, orange, yellow, blue, lightblue, grey))

plt.xlabel("Heating Power [W/m²]")
plt.title('Breakdown of the zone Heat Loads [W/m²]')
plt.legend(ncol = 1, loc="upper left",
          bbox_to_anchor=(1, 0, 0.5, 1))
plt.grid(linestyle = 'dotted');

In [ ]:
with pd.ExcelWriter(project + "_output.xlsx") as writer: 

    # DATA
    
    # General data
    df_gen_data.to_excel(writer, sheet_name='general_data')

    # Reheat data
    df_rh_data.to_excel(writer, sheet_name='reheat_data') 

    # Ventilation data
    dfv.to_excel(writer, sheet_name='ventilation_data') 

    # Materials data
    materials.to_excel(writer, sheet_name='materials_data')

    # Wall types layers data
    wt_layers.to_excel(writer, sheet_name='wall_types_layers_data')

    # Wall types environmental data
    wt_env.to_excel(writer, sheet_name='wall_types_env_data')

    # Walls data
    wl.to_excel(writer, sheet_name='wall_data')

    # Windows data
    wd.to_excel(writer, sheet_name='windows_data')

    # Doors data
    dr.to_excel(writer, sheet_name='doors_data')
    

    # RESULTS

    # dfwt : Walls thicknesses [m] and conductive resistance [m²K/W]
    dfwt.round(2).to_excel(writer, sheet_name='walls_R') 
    
    # dfAU  : Walls U_values [W/m²K] - Temperature Factors - Areas [m²] - Heat Tranfer Coefficients [W/K]
    dfAU.round(2).to_excel(writer, sheet_name='walls_AU') 

    # dfgrAU : Walls Areas [m²] and Heat Tranfer Coefficients [W/K] by zone and wall type
    dfgrAU.round(2).round(2).to_excel(writer, sheet_name='walls_AU_grouped') 

    # dfAHT : Wall Areas [m²] and Heat Tranfer Coefficients [W/K] by zone
    dfAHT.round(2).to_excel(writer, sheet_name='HT_by_zone') 

    # dfH   : Total Heat Tranfer Coefficients by zone [W/K]
    dfH.round(2).to_excel(writer, sheet_name='HTOT_by_zone') 

    # dfPHI : Emission and production Heat losses and Loads by zone [W]
    dfPHI.astype(int).to_excel(writer, sheet_name='HL_by_zone') 

    # dffv  : Emission Heat losses and loads by zone and by facade [W]
    dffv.astype(int).to_excel(writer, sheet_name='HL_by_zone_by_facade') 

    # dffPHI : Emission Heat losses and loads by facade [W]
    dffPHI.astype(int).to_excel(writer, sheet_name='HL_by_facade') 